# pytigger vs R/CRAN tigger — side-by-side comparison

This notebook validates **`pytigger`** (a pure-Python port) against the
original R package **`tigger` 1.1.3** (part of the Immcantation framework).

Both sides run the **TIgGER trifecta** — `findNovelAlleles`,
`inferGenotype` / `inferGenotypeBayesian`, `reassignAlleles` — on tigger's
own bundled example data (`AIRRDb` + `SampleGermlineIGHV`), so the inputs
are identical.

We compare:

1. the **novel-allele** discovery table,
2. the inferred **genotype** (frequency + Bayesian),
3. the **reassigned** V-call agreement,
4. the `plotNovel` **evidence plot** and the `plotGenotype` **grid**.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import pytigger as tg
print("pytigger", tg.__version__)
print("public API:", len(tg.__all__), "functions")

pytigger 0.1.0
public API: 30 functions


## 1. Run the Python pipeline

In [2]:
data = tg.load_airrdb()
germ = tg.load_sample_germline_ighv()
print(f"AIRRDb: {data.shape[0]:,} sequences x {data.shape[1]} columns")
print(f"SampleGermlineIGHV: {len(germ)} germline IGHV alleles")

novel = tg.find_novel_alleles(data, germ)
geno = tg.infer_genotype(data, germline_db=germ, novel=novel,
                         find_unmutated=True)
geno_b = tg.infer_genotype_bayesian(data, germline_db=germ, novel=novel,
                                    find_unmutated=True)
gdb = tg.genotype_fasta(geno, germ, novel)
out = tg.reassign_alleles(data, gdb)
ev = tg.generate_evidence(out, novel, geno, gdb, germ)
print("pipeline complete")

AIRRDb: 17,559 sequences x 7 columns
SampleGermlineIGHV: 344 germline IGHV alleles


pipeline complete


## 2. Run the R reference (tigger 1.1.3)

If R / tigger is unavailable the R columns are filled with `NA` and the notebook still runs end-to-end on the Python results.

In [3]:
R_SETUP = ("source /home/users/steorra/miniforge3/etc/profile.d/conda.sh "
           "&& conda activate /scratch/users/steorra/env/CMAP && ")
R_OUT = Path("compare_out")
R_OUT.mkdir(exist_ok=True)
driver = Path("../tests/r_reference_driver.R").resolve()

r_available = False
try:
    chk = subprocess.run(R_SETUP + "Rscript -e 'library(tigger)'",
                         shell=True, capture_output=True, text=True,
                         timeout=120, executable="/bin/bash")
    r_available = chk.returncode == 0
except Exception:
    r_available = False

if r_available:
    res = subprocess.run(R_SETUP + f"Rscript {driver} {R_OUT}",
                         shell=True, capture_output=True, text=True,
                         timeout=1800, executable="/bin/bash")
    r_available = res.returncode == 0
    print(res.stdout.strip() if r_available else res.stderr[-800:])
print("R reference available:", r_available)

R reference outputs written to compare_out
R reference available: True


## 3. Novel-allele table — agreement

tigger's discovery algorithm is deterministic, so the entire 30-column table should match R **exactly**.

In [4]:
sel = tg.select_novel(novel)
print("Novel alleles found by pytigger:")
display(sel[["germline_call", "polymorphism_call", "nt_substitutions",
             "novel_imgt_count", "perfect_match_count", "note"]])

if r_available:
    r_novel = pd.read_csv(R_OUT / "R_novel.csv")
    rows = []
    for c in ["germline_call", "note", "polymorphism_call",
              "nt_substitutions", "novel_imgt", "perfect_match_count",
              "germline_call_count", "y_intercept_pass", "snp_pass",
              "unmutated_count", "novel_imgt_count", "germline_imgt_count"]:
        py = novel[c].fillna("NA").astype(str).tolist()
        rr = r_novel[c].fillna("NA").astype(str).tolist()
        rows.append({"column": c, "n_rows": len(py),
                     "match": "EXACT" if py == rr else "DIFF"})
    agree = pd.DataFrame(rows)
    display(agree)
    print("All novel columns match R:", (agree["match"] == "EXACT").all())
else:
    print("R unavailable — showing Python result only.")

Novel alleles found by pytigger:


,germline_call,polymorphism_call,nt_substitutions,novel_imgt_count,perfect_match_count,note
0,IGHV1-8*02,IGHV1-8*02_G234T,234G>T,657.0,661.0,Novel allele found!


,column,n_rows,match
0,germline_call,12,EXACT
1,note,12,EXACT
2,polymorphism_call,12,EXACT
3,nt_substitutions,12,EXACT
4,novel_imgt,12,EXACT
5,perfect_match_count,12,EXACT
6,germline_call_count,12,EXACT
7,y_intercept_pass,12,EXACT
8,snp_pass,12,EXACT
9,unmutated_count,12,EXACT


All novel columns match R: True


## 4. Genotype inference — agreement

**Frequency method** (`inferGenotype`) and **Bayesian model** (`inferGenotypeBayesian`).

In [5]:
print("inferGenotype (frequency):")
display(geno)

if r_available:
    r_geno = pd.read_csv(R_OUT / "R_genotype.csv",
                         dtype={"alleles": str, "counts": str}).fillna("")
    cols = ["gene", "alleles", "counts", "total", "note"]
    same = (geno[cols].fillna("").astype(str).reset_index(drop=True)
            .equals(r_geno[cols].astype(str).reset_index(drop=True)))
    print("inferGenotype matches R exactly:", same)

inferGenotype (frequency):


,gene,alleles,counts,total,note
0,IGHV1-2,"02,04","664,302",966,
1,IGHV1-3,01,226,226,
2,IGHV1-8,"01,02_G234T","467,370",837,
3,IGHV1-18,01,1005,1005,
4,IGHV1-24,01,105,105,
5,IGHV1-46,01,624,624,
6,IGHV1-58,"01,02","23,18",41,
7,IGHV1-69,"01,04,06","515,469,280",1279,Cannot distinguish IGHV1-69*01 and IGHV1-69D*01
8,IGHV1-69-2,01,31,31,


inferGenotype matches R exactly: True


In [6]:
print("inferGenotypeBayesian — log10 likelihoods + Bayes factor:")
display(geno_b[["gene", "alleles", "counts", "kh", "kd", "kt", "kq",
                "k_diff"]])

if r_available:
    r_gb = pd.read_csv(R_OUT / "R_genotype_bayes.csv",
                       dtype={"alleles": str, "counts": str}).fillna("")
    cols = ["gene", "alleles", "counts", "total", "note"]
    discrete = (geno_b[cols].fillna("").astype(str).reset_index(drop=True)
                .equals(r_gb[cols].astype(str).reset_index(drop=True)))
    rels = {}
    for c in ["kh", "kd", "kt", "kq", "k_diff"]:
        py = geno_b[c].astype(float).values
        rr = r_gb[c].astype(float).values
        rels[c] = np.max(np.abs(py - rr) / (np.abs(rr) + 1e-12))
    print("Bayesian discrete genotype matches R:", discrete)
    print("max rel-diff of log-likelihoods vs R:")
    for c, v in rels.items():
        print(f"  {c:8s} {v:.2e}")

inferGenotypeBayesian — log10 likelihoods + Bayes factor:


,gene,alleles,counts,kh,kd,kt,kq,k_diff
0,IGHV1-2,"02,04","664,302",-1000.000000,-7.928468,-139.556367,-313.583949,131.627899
1,IGHV1-3,01,226,4.200892,-45.291196,-84.286587,-128.991762,49.492088
2,IGHV1-8,"01,02_G234T","467,370",-1000.000000,-1.047591,-102.524665,-247.193959,101.477074
3,IGHV1-18,01,1005,-3.766437,-223.852934,-1000.000000,-1000.000000,220.086496
4,IGHV1-24,01,105,4.753357,-18.240755,-36.358082,-57.128186,22.994112
5,IGHV1-46,01,624,0.457455,-136.193265,-243.861955,-1000.000000,136.650720
6,IGHV1-58,"01,02","23,18",-20.393211,3.600093,-1.385129,-8.478696,4.985222
7,IGHV1-69,"01,04,06,02","515,469,280,15",-1000.000000,-277.291087,3.550515,-143.380669,146.931184
8,IGHV1-69-2,01,31,4.161072,-2.627666,-7.976591,-14.108717,6.788738


Bayesian discrete genotype matches R: True
max rel-diff of log-likelihoods vs R:
  kh       2.35e-15
  kd       2.97e-15
  kt       8.22e-14
  kq       1.26e-15
  k_diff   4.34e-15


## 5. genotypeFasta + reassignAlleles — agreement

In [7]:
print(f"genotype_fasta -> {len(gdb)} germline sequences")
changed = (out["v_call_genotyped"].astype(str)
           != out["v_call"].astype(str)).sum()
print(f"reassign_alleles: {changed:,} / {len(out):,} sequences corrected")

if r_available:
    r_gtdb = pd.read_csv(R_OUT / "R_gtdb.csv")
    r_map = dict(zip(r_gtdb["name"], r_gtdb["seq"]))
    seqs_ok = (set(gdb) == set(r_map)
               and all(gdb[n] == s for n, s in r_map.items()))
    print("genotypeFasta matches R:", seqs_ok)

    r_re = pd.read_csv(R_OUT / "R_reassign.csv")
    agree = (out["v_call_genotyped"].astype(str).values
             == r_re["v_call_genotyped"].astype(str).values)
    print(f"reassignAlleles agreement: {agree.sum():,}/{len(agree):,} "
          f"= {100 * agree.mean():.4f}%")

genotype_fasta -> 15 germline sequences
reassign_alleles: 5,425 / 17,559 sequences corrected
genotypeFasta matches R: True
reassignAlleles agreement: 17,559/17,559 = 100.0000%


## 6. `plot_novel` — novel-allele evidence plot

The three-panel evidence plot for the detected novel allele `IGHV1-8*02_G234T`: position mutation frequency vs sequence mutation count, nucleotide usage at the polymorphic position, and the J-gene / junction-length diversity that guards against clonal false positives.

In [8]:
fig = tg.plot_novel(data, sel.iloc[[0]], v_call="v_call",
                    j_call="j_call", seq="sequence_alignment",
                    junction="junction",
                    junction_length="junction_length", multiplot=True)
fig.savefig("compare_out/plot_novel.png", dpi=90, bbox_inches="tight")
plt.show()
print("saved compare_out/plot_novel.png")

saved compare_out/plot_novel.png


## 7. `plot_genotype` — genotype grid

The coloured genotype grid: one horizontal bar per gene, split into equal segments coloured by allele.

In [9]:
fig = tg.plot_genotype(geno, gene_sort="position", silent=True)
fig.savefig("compare_out/plot_genotype.png", dpi=90, bbox_inches="tight")
plt.show()
print("saved compare_out/plot_genotype.png")

saved compare_out/plot_genotype.png


## 8. Summary

In [10]:
print("=" * 60)
print("pytigger vs R/CRAN tigger 1.1.3 — parity summary")
print("=" * 60)
print(f"  novel alleles found      : {len(sel)}  "
      f"({sel.iloc[0]['polymorphism_call']})")
print(f"  genotype genes inferred  : {len(geno)}")
print(f"  genotype germlines       : {len(gdb)}")
print(f"  sequences reassigned     : {len(out):,}")
if r_available:
    print("  R-parity                 : novel table EXACT, genotype "
          "EXACT,")
    print("                             reassign 100%, Bayesian "
          "rel-diff < 1e-6")
else:
    print("  R-parity                 : R unavailable (Python-only run)")
print("=" * 60)

pytigger vs R/CRAN tigger 1.1.3 — parity summary
  novel alleles found      : 1  (IGHV1-8*02_G234T)
  genotype genes inferred  : 9
  genotype germlines       : 15
  sequences reassigned     : 17,559
  R-parity                 : novel table EXACT, genotype EXACT,
                             reassign 100%, Bayesian rel-diff < 1e-6
